In [ ]:
import os
import sys
import numpy as np
import pyvista as pv

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

from src.FibGenOO import FibGenBayer
from src.SurfaceNames import SurfaceName

# Configure PyVista for Jupyter
pv.set_jupyter_backend('static')

# Define paths (update these paths as needed)
data_dir = "example/truncated"
mesh_path = os.path.join(data_dir, "VOLUME.vtu")
laplace_results_file = os.path.join(data_dir, "output_b_oo", "result_001.vtu")

# Bayer method parameters
params = {
    "ALFA_END": 60.0,    # Endocardial helix angle (degrees)
    "ALFA_EPI": -60.0,   # Epicardial helix angle (degrees)
    "BETA_END": 20.0,    # Endocardial transverse angle (degrees)
    "BETA_EPI": -20.0,   # Epicardial transverse angle (degrees)
}

# Load the mesh
mesh = pv.read(mesh_path)# Initialize fiber generator

# Initialize fiber generator
fib_gen = FibGenBayer()
self = fib_gen  # For easier reference in this context
fib_gen.load_laplace_results(laplace_results_file)

# Convert parameters to radians (consistent with Doste method)
params = {k: np.deg2rad(v) for k, v in params.items()}

## Step 1: Calculate Q_LV_endo and Q_RV_endo
The FibGenBayer class handles the fiber generation pipeline. First, we load the pre-computed Laplace field solutions.

In [ ]:
# Interpolation factor between LV and RV
d = self.lap['Trans_RV'] / (self.lap['Trans_LV'] + self.lap['Trans_RV'])

# Septum angles (interpolated between LV and RV)
alfaS = self.calculate_angle(d, params['ALFA_END'], -params['ALFA_END'])
betaS = self.calculate_angle(d, params['BETA_END'], -params['BETA_END'])

# Wall angles (interpolated from endo to epi)
alfaW = self.calculate_angle(self.lap['Trans_EPI'], 0, params['ALFA_EPI'])
betaW = self.calculate_angle(self.lap['Trans_EPI'], 0, params['BETA_EPI'])

# Build LV and RV basis
Q_LV0 = self.calculate_basis(self.grad['Long_AB'], -self.grad['Trans_LV'])
Q_LV = self.rotate_basis_matrix(Q_LV0, alfaS, betaS)

Q_RV0 = self.calculate_basis(self.grad['Long_AB'], self.grad['Trans_RV'])
Q_RV = self.rotate_basis_matrix(Q_RV0, alfaS, -betaS)

In [ ]:
# Visualize Q_LV circumferential direction as glyphs
p = pv.Plotter()

# Sample every 15th element to avoid clutter
sample_factor = 15
sampled_mesh = fib_gen.mesh.extract_cells(np.arange(0, fib_gen.mesh.n_cells, sample_factor))

# Add Q_LV circumferential direction to mesh cell data
sampled_Q_LV_circ = Q_LV[::sample_factor, :, 0].copy()
sampled_mesh.cell_data['Q_LV_circ'] = sampled_Q_LV_circ

# Create glyph representation
arrows = sampled_mesh.glyph(orient='Q_LV_circ', scale=False, factor=0.02)

# Plot
p.add_mesh(fib_gen.mesh, color='lightgray', opacity=0.3, label='Mesh')
p.add_mesh(arrows, color='red', opacity=0.9, label='Q_LV[:,:,0] (circumferential)')

p.add_legend()
p.camera_position = 'xy'
p.show()

Traceback (most recent call last):
  File "/home/javiera/.vscode/extensions/ms-python.python-2026.0.0-linux-x64/python_files/python_server.py", line 134, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 10, in <module>
  File "/home/javiera/miniconda3/envs/main/lib/python3.12/site-packages/pyvista/_deprecate_positional_args.py", line 245, in inner_f
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/home/javiera/miniconda3/envs/main/lib/python3.12/site-packages/pyvista/core/filters/data_set.py", line 1798, in glyph
    if orient:
       ^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()



## Step 3: Visualize Laplace Fields

Before generating fibers, let's visualize the Laplace field solutions to understand the transmural and interventricular coordinates.

In [ ]:
# Create subplots for the Laplace fields
p = pv.Plotter(shape=(2, 2), figsize=(12, 10))

# Transmural field (endo->epi)
p.subplot(0, 0)
p.add_mesh(fib_gen.mesh, scalars='Trans_EPI', cmap='viridis', scalar_bar_args={'title': 'Transmural'})
p.camera_position = 'xy'

# LV chamber field
p.subplot(0, 1)
p.add_mesh(fib_gen.mesh, scalars='Trans_LV', cmap='plasma', scalar_bar_args={'title': 'LV Chamber'})
p.camera_position = 'xy'

# RV chamber field
p.subplot(1, 0)
p.add_mesh(fib_gen.mesh, scalars='Trans_RV', cmap='plasma', scalar_bar_args={'title': 'RV Chamber'})
p.camera_position = 'xy'

# Apex-to-base field
p.subplot(1, 1)
p.add_mesh(fib_gen.mesh, scalars='Long_AB', cmap='coolwarm', scalar_bar_args={'title': 'Apex-to-Base'})
p.camera_position = 'xy'

p.show()

## Step 4: Generate Fiber Directions

Now we run the main fiber generation algorithm. This step:
1. Constructs the local basis vectors (e_c, e_l, e_t) from gradients
2. Computes angle variations across the wall thickness and interventricular septum
3. Applies rotation transformations to orient fibers
4. Interpolates between LV and RV bases using spherical linear interpolation (SLERP)
5. Returns fiber, sheet, and sheet-normal directions

In [ ]:
print("Running Bayer fiber generation...")
print(f"Parameters: {params}\n")

# Generate fibers
F, S, T = fib_gen.generate_fibers(params)

print(f"Generated fiber directions:")
print(f"  Fiber (F) shape: {F.shape}")
print(f"  Sheet (T) shape: {T.shape}")
print(f"  Sheet-normal (S) shape: {S.shape}")

# Verify orthonormality for a sample of elements
n_samples = 5
sample_indices = np.random.choice(len(F), n_samples, replace=False)

print(f"\nOrthonormality check (sample of {n_samples} elements):")
for idx in sample_indices:
    f_norm = np.linalg.norm(F[idx])
    t_norm = np.linalg.norm(T[idx])
    s_norm = np.linalg.norm(S[idx])
    f_dot_t = np.dot(F[idx], T[idx])
    f_dot_s = np.dot(F[idx], S[idx])
    t_dot_s = np.dot(T[idx], S[idx])
    print(f"  Element {idx}: |F|={f_norm:.4f}, |T|={t_norm:.4f}, |S|={s_norm:.4f}")
    print(f"              F·T={f_dot_t:.6f}, F·S={f_dot_s:.6f}, T·S={t_dot_s:.6f}")

## Step 5: Visualize Fiber Orientation

Display the mesh colored by fiber direction components and overlay with glyph arrows showing the fiber vectors.

In [ ]:
# Visualize fiber directions - colored by helix angle
p = pv.Plotter(figsize=(14, 5))

# Get helix angle for coloring
helix_angle = fib_gen.mesh.cell_data.get('alfaS', np.zeros(fib_gen.mesh.n_cells))

# Plot mesh colored by helix angle
p.add_mesh(fib_gen.mesh, scalars='alfaS', cmap='RdBu_r', scalar_bar_args={'title': 'Helix Angle (rad)'})

# Add fiber direction glyphs (sample every 10th element to avoid clutter)
sample_factor = 10
sampled_mesh = fib_gen.mesh.extract_cells(np.arange(0, fib_gen.mesh.n_cells, sample_factor))
sampled_fiber = F[::sample_factor]

# Create glyph representation of fiber directions
arrows = sampled_mesh.glyph(orient=sampled_fiber, scale=False, factor=0.02)
p.add_mesh(arrows, color='black', opacity=0.8)

p.camera_position = 'xy'
p.show()

In [ ]:
# Visualize sheet and sheet-normal directions in subplots
p = pv.Plotter(shape=(1, 2), figsize=(14, 6))

# Fiber direction
p.subplot(0, 0)
sampled_mesh = fib_gen.mesh.extract_cells(np.arange(0, fib_gen.mesh.n_cells, 15))
sampled_fiber = F[::15]
arrows_f = sampled_mesh.glyph(orient=sampled_fiber, scale=False, factor=0.015)
p.add_mesh(fib_gen.mesh, color='lightgray', opacity=0.5)
p.add_mesh(arrows_f, color='red', opacity=0.9)
p.set_title('Fiber Direction (F)')
p.camera_position = 'xy'

# Sheet direction
p.subplot(0, 1)
sampled_sheet = T[::15]
arrows_s = sampled_mesh.glyph(orient=sampled_sheet, scale=False, factor=0.015)
p.add_mesh(fib_gen.mesh, color='lightgray', opacity=0.5)
p.add_mesh(arrows_s, color='blue', opacity=0.9)
p.set_title('Sheet Direction (T)')
p.camera_position = 'xy'

p.show()

## Step 6: Analyze Intermediate Basis Vectors

The algorithm constructs multiple basis systems at different stages. Let's visualize these intermediate steps.

In [ ]:
# Extract intermediate basis vectors from mesh cell data
eC_LV = fib_gen.mesh.cell_data.get('eC_LV', None)
eL_LV = fib_gen.mesh.cell_data.get('eL_LV', None)
eT_LV = fib_gen.mesh.cell_data.get('eT_LV', None)

eC_RV = fib_gen.mesh.cell_data.get('eC_RV', None)
eL_RV = fib_gen.mesh.cell_data.get('eL_RV', None)
eT_RV = fib_gen.mesh.cell_data.get('eT_RV', None)

eC_END = fib_gen.mesh.cell_data.get('eC_END', None)
eL_END = fib_gen.mesh.cell_data.get('eL_END', None)
eT_END = fib_gen.mesh.cell_data.get('eT_END', None)

print("Intermediate basis data available:")
print(f"  LV basis: {eC_LV is not None}")
print(f"  RV basis: {eC_RV is not None}")
print(f"  Endocardial basis: {eC_END is not None}")

# Visualize LV and RV bases side by side
if eC_LV is not None and eC_RV is not None:
    p = pv.Plotter(shape=(1, 2), figsize=(14, 6))
    
    sample_factor = 20
    sampled_mesh = fib_gen.mesh.extract_cells(np.arange(0, fib_gen.mesh.n_cells, sample_factor))
    
    # LV basis
    p.subplot(0, 0)
    p.add_mesh(fib_gen.mesh, color='lightgray', opacity=0.3)
    
    # LV longitudinal (green), circumferential (red), transmural (blue)
    arrows_eL = sampled_mesh.glyph(orient=eL_LV[::sample_factor], scale=False, factor=0.015)
    p.add_mesh(arrows_eL, color='green', opacity=0.7)
    
    p.set_title('LV Basis (Longitudinal)')
    p.camera_position = 'xy'
    
    # RV basis
    p.subplot(0, 1)
    p.add_mesh(fib_gen.mesh, color='lightgray', opacity=0.3)
    
    arrows_eL_rv = sampled_mesh.glyph(orient=eL_RV[::sample_factor], scale=False, factor=0.015)
    p.add_mesh(arrows_eL_rv, color='green', opacity=0.7)
    
    p.set_title('RV Basis (Longitudinal)')
    p.camera_position = 'xy'
    
    p.show()

## Step 7: Visualize Angle Fields

The Bayer method varies the helix (α) and transverse (β) angles across the wall and interventricular septum.

In [ ]:
# Create angle field visualizations
if 'alfaS' in fib_gen.mesh.cell_data and 'betaS' in fib_gen.mesh.cell_data:
    p = pv.Plotter(shape=(2, 2), figsize=(14, 12))
    
    # Septum angles
    p.subplot(0, 0)
    p.add_mesh(fib_gen.mesh, scalars='alfaS', cmap='RdBu_r', 
               scalar_bar_args={'title': 'Helix Angle (septum, α_s)'})
    p.camera_position = 'xy'
    
    p.subplot(0, 1)
    p.add_mesh(fib_gen.mesh, scalars='betaS', cmap='RdBu_r',
               scalar_bar_args={'title': 'Transverse Angle (septum, β_s)'})
    p.camera_position = 'xy'
    
    # Wall angles
    p.subplot(1, 0)
    p.add_mesh(fib_gen.mesh, scalars='alfaW', cmap='RdBu_r',
               scalar_bar_args={'title': 'Helix Angle (wall, α_w)'})
    p.camera_position = 'xy'
    
    p.subplot(1, 1)
    p.add_mesh(fib_gen.mesh, scalars='betaW', cmap='RdBu_r',
               scalar_bar_args={'title': 'Transverse Angle (wall, β_w)'})
    p.camera_position = 'xy'
    
    p.show()

## Step 8: Parametric Study - Varying Angle Parameters

Let's explore how changes to the angle parameters affect the fiber orientations.

In [ ]:
# Test different parameter sets
param_sets = {
    'Default (α: 60°/-60°, β: 20°/-20°)': {
        "ALFA_END": 60.0,
        "ALFA_EPI": -60.0,
        "BETA_END": 20.0,
        "BETA_EPI": -20.0,
    },
    'Zero angles': {
        "ALFA_END": 0.0,
        "ALFA_EPI": 0.0,
        "BETA_END": 0.0,
        "BETA_EPI": 0.0,
    },
    'Increased helix (α: 80°/-80°)': {
        "ALFA_END": 80.0,
        "ALFA_EPI": -80.0,
        "BETA_END": 20.0,
        "BETA_EPI": -20.0,
    },
}

results = {}

for name, params_test in param_sets.items():
    print(f"\nGenerating fibers with parameters: {name}")
    
    # Create a fresh fiber generator with the same mesh
    fib_gen_test = FibGenBayer()
    fib_gen_test.load_laplace_results(laplace_results_file)
    
    # Generate fibers
    F_test, S_test, T_test = fib_gen_test.generate_fibers(params_test)
    results[name] = (F_test, S_test, T_test)
    
    print(f"  Generated: F shape {F_test.shape}")

print("\nAll parameter sets generated successfully!")

In [ ]:
# Visualize results from different parameter sets
p = pv.Plotter(shape=(1, len(results)), figsize=(5*len(results), 5))

for idx, (name, (F_test, S_test, T_test)) in enumerate(results.items()):
    p.subplot(0, idx)
    
    # Sample and create glyph
    sample_factor = 20
    sampled_mesh = fib_gen.mesh.extract_cells(np.arange(0, fib_gen.mesh.n_cells, sample_factor))
    sampled_fiber = F_test[::sample_factor]
    
    arrows = sampled_mesh.glyph(orient=sampled_fiber, scale=False, factor=0.015)
    p.add_mesh(fib_gen.mesh, color='lightgray', opacity=0.3)
    p.add_mesh(arrows, color='red', opacity=0.8)
    
    p.set_title(name, font_size=10)
    p.camera_position = 'xy'

p.show()

## Step 9: Export Results to VTU File

Save the fiber directions and related data to a VTU file for visualization in ParaView or other software.

In [ ]:
# The mesh already has the results saved from generate_fibers
# The check.vtu file was automatically created during the process
check_file = os.path.abspath("../check.vtu")

if os.path.exists(check_file):
    print(f"Results saved to: {check_file}")
    
    # Read and verify the saved file
    saved_mesh = pv.read(check_file)
    print(f"\nSaved mesh contains:")
    print(f"  Cell data: {list(saved_mesh.cell_data.keys())}")
    
    # Can be visualized in ParaView
    print("\nTo visualize in ParaView:")
    print(f"  paraview {check_file} &")
else:
    print(f"Note: {check_file} not found")
    print("The check.vtu file is created during generate_fibers execution.")

## Summary

The Bayer fiber generation method consists of the following key steps:

1. **Laplace Fields**: Solve Laplace equations to obtain transmural (endo↔epi), interventricular (LV↔RV), and apex-to-base coordinate fields.

2. **Basis Vectors**: Construct orthonormal bases from field gradients at each location.

3. **Angle Variation**: Define helix (α) and transverse (β) angles that vary across the wall and septum.

4. **Rotation**: Apply rotation matrices (or Rodrigues' formula) to orient the basis vectors.

5. **Interpolation**: Use spherical linear interpolation (SLERP) to smoothly blend between LV and RV bases, and between endocardial and epicardial layers.

6. **Output**: Extract fiber, sheet, and sheet-normal directions for each element.

The resulting fiber orientations exhibit the characteristic helical pattern observed in cardiac tissue, with transmural variation in helix angle and regional differences between the left and right ventricles.

### References
- Bayer et al. 2012: https://doi.org/10.1007/s10439-012-0593-5
- Documentation: See `../DOCUMENTATION.md` for detailed mathematical formulation